# Auditoria executável do pipeline

**Regra central:** célula que não pode falhar não é auditoria. Toda verificação é um
`assert` sobre o **DataFrame completo** — nunca uma amostra, nunca um print para inspeção
visual. Tabelas existem como evidência *depois* do assert que prova a regra.

Toda a lógica vem de `src/` (`io`, `clean`, `attribution`, `features`, `cost`, `contract`,
`pipeline`); nada é reimplementado aqui — auditar uma cópia não prova nada sobre o produto.
Os parâmetros do pipeline vêm de `src/config.py`.

Regras auditadas: garantias **G1–G9** (`specification/schema-processed.md`) e requisitos
**REQ-101 … REQ-110** (`specification/spec.md`).

<a id="s0"></a>
## S0 — Setup e reconciliação de contagens

**Regra.** Cada `offer received` gera exatamente uma linha do grão, e nenhuma transação é
atribuída a mais de uma oferta.
**Prova.** Roda o pipeline estágio a estágio e assere a conservação de linhas em cada
fronteira, mais a reconciliação `transações elegíveis == Σ assigned_txn_count`.

In [1]:
import logging, os, re, sys
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Raiz do repo = primeiro ancestral com config.yaml (sem caminho literal).
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src import contract
from src.attribution import attribute, build_label
from src.clean import normalize_profile
from src.config import load
from src.cost import add_reward_cost
from src.features import build as build_features
from src.io import parse_events, read_events, read_offers, read_profile
from src.pipeline import assemble_processed, build_spark

cfg = load()  # todo parâmetro de comportamento vem daqui (REQ-110)

# Captura o log de premissa emitido por attribution.attribute (usado em S4).
class _LogCapture(logging.Handler):
    def __init__(self):
        super().__init__()
        self.messages: list[str] = []

    def emit(self, record):
        self.messages.append(record.getMessage())

_capture = _LogCapture()
_attr_logger = logging.getLogger("src.attribution")
_attr_logger.addHandler(_capture)
_attr_logger.setLevel(logging.WARNING)

# A sessão vem de src/pipeline.py, configurada pela cfg — nenhum literal aqui.
# Os caches são MEMORY_AND_DISK: derramam para disco em vez de estourar a heap.
spark = build_spark(cfg, app_name="pipeline-audit")
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| config:", cfg.model_dump())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/09 00:33:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.2 | config: {'raw_dir': PosixPath('data/raw'), 'processed_dir': PosixPath('data/processed'), 'offers_filename': 'offers.json', 'profile_filename': 'profile.json', 'transactions_filename': 'transactions.json', 'age_sentinel': 118, 'test_start_date': '20180726', 'smd_threshold': 0.1, 'attribution_priority': <AttributionPriority.EARLIEST_RECEIVED: 'earliest_received'>, 'n_campaign_waves': 6, 'contract_sample_size': 1000, 'spark_master': 'local[*]', 'spark_driver_memory': '4g', 'spark_shuffle_partitions': 16, 'seed': 42}


In [2]:
# --- Harness de auditoria -----------------------------------------------------
# check() registra E impõe. Uma regra violada levanta na hora: o notebook fica
# vermelho na célula que a provou. Nenhum try/except engole falha.

AUDIT: list[dict] = []
SEC = "S0"


def check(rule: str, claim: str, ok, evidence: str = "") -> None:
    ok = bool(ok)
    AUDIT.append({"regra": rule, "seção": SEC, "afirmação": claim,
                  "status": "PASS" if ok else "FAIL", "evidência": evidence})
    if not ok:
        raise AssertionError(f"[{rule}] VIOLADA — {claim} | {evidence}")


# Fatos do dataset que a auditoria fixa (não são parâmetros do pipeline: são o que
# o dado bruto contém). Se o dado mudar, a auditoria deve falhar e ser revista.
FATOS = {
    "raw_events": 306_534,
    "offer_received": 76_277,
    "offer_viewed": 57_725,
    "offer_completed": 33_579,
    "transactions": 138_953,
    "clients": 17_000,
    "identity_missing": 2_175,
}

In [3]:
# --- Estágios do pipeline, na ordem real --------------------------------------
raw = read_events(spark, cfg)
events = parse_events(spark, cfg).cache()
offers = read_offers(spark, cfg).cache()
profile = normalize_profile(read_profile(spark, cfg), cfg)

attributed = attribute(events, offers, cfg).cache()
labeled = build_label(attributed, cfg).cache()
featured = build_features(events, labeled, offers, cfg)
priced = add_reward_cost(featured, offers, cfg)
processed = assemble_processed(events, offers, profile, cfg).cache()

n_raw = raw.count()
por_tipo = {r["event"]: r["n"] for r in events.groupBy("event").agg(F.count("*").alias("n")).collect()}
n_received = por_tipo["offer received"]
n_txn = por_tipo["transaction"]
n_grain = processed.count()

# FATO = propriedade do dado bruto que a auditoria fixa (não é uma regra do contrato).
# Se o dado de entrada mudar, estas falham primeiro e o resto da auditoria é revisto.
check("FATO", "o JSON bruto tem exatamente os eventos esperados",
      n_raw == FATOS["raw_events"], f"{n_raw:,} eventos brutos")
check("FATO", "a composição por tipo de evento é a esperada",
      (por_tipo["offer received"] == FATOS["offer_received"]
       and por_tipo["offer viewed"] == FATOS["offer_viewed"]
       and por_tipo["offer completed"] == FATOS["offer_completed"]
       and por_tipo["transaction"] == FATOS["transactions"]),
      f"{por_tipo}")
check("G1", "cada `offer received` gera exatamente uma linha do grão",
      n_grain == n_received, f"{n_received:,} received → {n_grain:,} linhas no grão")

In [4]:
# Reconciliação de transações: nenhuma é atribuída duas vezes, nenhuma some.
# "Elegível" = cai na janela pós-view de pelo menos um recebimento.
txns = (
    events.filter(F.col("event") == "transaction")
    .select("account_id", F.col("time").alias("txn_time"), F.col("amount").alias("txn_amount"))
    .withColumn("row_id", F.monotonically_increasing_id())
    .cache()
)
candidates = attributed.join(txns, on="account_id", how="inner").filter(
    (F.col("txn_time") >= F.col("view_time")) & (F.col("txn_time") <= F.col("valid_until"))
).cache()

n_eligible = candidates.select("row_id").distinct().count()
n_assigned = labeled.agg(F.sum("assigned_txn_count")).first()[0]
n_unassigned = n_txn - n_assigned

check("REQ-103", "toda transação elegível é atribuída a exatamente uma oferta",
      n_eligible == n_assigned, f"elegíveis={n_eligible} == Σ assigned={n_assigned}")
check("REQ-103", "nenhuma transação é atribuída mais de uma vez",
      n_assigned <= n_txn, f"Σ assigned={n_assigned} ≤ total={n_txn}")

pd.DataFrame([
    {"estágio": "raw (transactions.json)", "linhas": n_raw},
    {"estágio": "offer received", "linhas": n_received},
    {"estágio": "attribute → grão", "linhas": attributed.count()},
    {"estágio": "build_label", "linhas": labeled.count()},
    {"estágio": "features.build", "linhas": featured.count()},
    {"estágio": "cost.add_reward_cost", "linhas": priced.count()},
    {"estágio": "contrato (processed)", "linhas": n_grain},
])

,estágio,linhas
0,raw (transactions.json),306534
1,offer received,76277
2,attribute → grão,76277
3,build_label,76277
4,features.build,76277
5,cost.add_reward_cost,76277
6,contrato (processed),76277


In [5]:
# As não-atribuídas são compras sem nenhuma oferta vista e ativa no momento —
# exatamente o comportamento de compra "orgânica" que o uplift precisa isolar.
print(f"transações totais .......... {n_txn:,}")
print(f"  atribuídas (pós-view) .... {n_assigned:,} ({100*n_assigned/n_txn:.1f}%)")
print(f"  não-atribuídas ........... {n_unassigned:,} ({100*n_unassigned/n_txn:.1f}%)")
print("\nNão-atribuída = transação fora de qualquer janela [view_time, valid_until].")

transações totais .......... 138,953
  atribuídas (pós-view) .... 102,326 (73.6%)
  não-atribuídas ........... 36,627 (26.4%)

Não-atribuída = transação fora de qualquer janela [view_time, valid_until].


<a id="s1"></a>
## S1 — Parsing de `value` (REQ-101)

**Regra.** `received`/`viewed` referenciam a oferta em `offer id` (com espaço), `completed`
em `offer_id` (com underscore), e `transaction` traz `amount` sem referência de oferta.
**Prova.** Assert por tipo de evento na base completa: zero referências nulas onde não podem
ser nulas, e zero referências presentes onde não podem existir.

In [6]:
SEC = "S1"
value = F.col("value")
espaco, underscore = value.getField("offer id"), value.getField("offer_id")

for evento in ("offer received", "offer viewed"):
    d = raw.filter(F.col("event") == evento)
    total = d.count()
    check("REQ-101", f"`{evento}` lê a oferta de `offer id` (com espaço), sempre presente",
          d.filter(espaco.isNull()).count() == 0, f"{total} eventos, 0 nulos em `offer id`")
    check("REQ-101", f"`{evento}` nunca usa `offer_id` (underscore)",
          d.filter(underscore.isNotNull()).count() == 0, f"{total} eventos, 0 usos de `offer_id`")

d = raw.filter(F.col("event") == "offer completed")
check("REQ-101", "`offer completed` lê a oferta de `offer_id` (underscore), sempre presente",
      d.filter(underscore.isNull()).count() == 0, f"{d.count()} eventos, 0 nulos em `offer_id`")
check("REQ-101", "`offer completed` nunca usa `offer id` (com espaço)",
      d.filter(espaco.isNotNull()).count() == 0, "0 usos de `offer id`")

d = raw.filter(F.col("event") == "transaction")
check("REQ-101", "`transaction` traz `amount` e nenhuma referência de oferta",
      d.filter(value.getField("amount").isNull()).count() == 0
      and d.filter(espaco.isNotNull() | underscore.isNotNull()).count() == 0,
      f"{d.count()} transações com amount, 0 com referência de oferta")

# A coalescência de io.parse_events preserva os dois nomes numa única `offer_ref`.
check("REQ-101", "`offer_ref` é não-nulo em todo evento de oferta e nulo em toda transação",
      events.filter((F.col("event") != "transaction") & F.col("offer_ref").isNull()).count() == 0
      and events.filter((F.col("event") == "transaction") & F.col("offer_ref").isNotNull()).count() == 0,
      "coalesce(`offer id`, `offer_id`) sem perda")

events.groupBy("event").agg(
    F.count("*").alias("eventos"),
    F.sum(F.col("offer_ref").isNotNull().cast("int")).alias("com_offer_ref"),
    F.sum(F.col("amount").isNotNull().cast("int")).alias("com_amount"),
).toPandas()

,event,eventos,com_offer_ref,com_amount
0,transaction,138953,0,138953
1,offer received,76277,76277,0
2,offer completed,33579,33579,0
3,offer viewed,57725,57725,0


<a id="s2"></a>
## S2 — Perfil e sentinela (REQ-102, G7)

**Regra.** `identity_missing=1` ⇔ (`age=118` ∧ `gender` nulo ∧ `credit_card_limit` nulo);
`age` nunca vale 118 após normalização; `tenure_days ≥ 0`.
**Prova.** Assert do bi-condicional linha a linha sobre os 17.000 clientes (zero violações do
XOR), mais asserts de domínio nas colunas derivadas.

In [7]:
SEC = "S2"
raw_profile = read_profile(spark, cfg).cache()

eh_sentinela = (
    (F.col("age") == cfg.age_sentinel)
    & F.col("gender").isNull()
    & F.col("credit_card_limit").isNull()
)
lado_a_lado = raw_profile.select(F.col("id").alias("account_id"), eh_sentinela.alias("bruto_sentinela")).join(
    profile.select("account_id", "identity_missing", "age", "tenure_days", "gender"), on="account_id"
).cache()

n_clientes = lado_a_lado.count()
n_flag = lado_a_lado.filter(F.col("identity_missing") == 1).count()
# XOR: a flag diverge do critério bruto em alguma linha?
n_xor = lado_a_lado.filter(F.col("bruto_sentinela") != (F.col("identity_missing") == 1)).count()

check("FATO", "o profile traz exatamente os clientes esperados",
      n_clientes == FATOS["clients"], f"{n_clientes:,} clientes")
check("G7", "identity_missing=1 ⇔ (age=118 ∧ gender nulo ∧ credit nulo), sem exceção",
      n_xor == 0, f"{n_clientes:,} clientes, 0 divergências do bi-condicional")
check("G7", "o segmento sentinela tem exatamente o tamanho esperado",
      n_flag == FATOS["identity_missing"], f"{n_flag} clientes com identity_missing=1")
check("G7", "`age` nunca vale a sentinela após a normalização",
      lado_a_lado.filter(F.col("age") == cfg.age_sentinel).count() == 0, "0 linhas com age=118")
check("REQ-102", "`tenure_days` é sempre não-negativo",
      lado_a_lado.filter(F.col("tenure_days") < 0).count() == 0, "0 linhas com tenure_days<0")
check("REQ-102", "`gender` ausente vira `unknown`, nunca nulo",
      lado_a_lado.filter(F.col("gender").isNull()).count() == 0, "0 nulos em gender")

lado_a_lado.groupBy("identity_missing").agg(
    F.count("*").alias("clientes"),
    F.sum(F.col("age").isNull().cast("int")).alias("age_nula"),
    F.sum((F.col("gender") == "unknown").cast("int")).alias("gender_unknown"),
).toPandas()

,identity_missing,clientes,age_nula,gender_unknown
0,1,2175,2175,2175
1,0,14825,0,0


<a id="s3"></a>
## S3 — Grão e unicidade (G1)

**Regra.** A chave `(account_id, offer_id, received_time)` é única — zero duplicatas.
**Prova.** Assert de contagem distinta == contagem total sobre o dataset processado completo.

In [8]:
SEC = "S3"
n_total = processed.count()
n_distinto = processed.select("account_id", "offer_id", "received_time").distinct().count()

check("G1", "a chave do grão não tem nenhuma duplicata",
      n_total == n_distinto, f"{n_total} linhas, {n_distinto} chaves distintas")

# A mesma oferta reenviada em ondas distintas gera linhas distintas (não colapsa).
reenvios = (
    processed.groupBy("account_id", "offer_id").agg(F.count("*").alias("n_recebimentos"))
    .filter(F.col("n_recebimentos") > 1)
)
print(f"pares (cliente, oferta) recebidos mais de uma vez: {reenvios.count():,}")
processed.select("campaign_wave", "received_time").groupBy("campaign_wave", "received_time") \
    .agg(F.count("*").alias("linhas")).orderBy("campaign_wave").toPandas()

pares (cliente, oferta) recebidos mais de uma vez: 11,718


,campaign_wave,received_time,linhas
0,0,0.0,12650
1,1,7.0,12669
2,2,14.0,12711
3,3,17.0,12778
4,4,21.0,12704
5,5,24.0,12765


In [9]:
# A onda é o rank do disparo distinto — deve haver exatamente n_campaign_waves ondas.
n_ondas = processed.select("campaign_wave").distinct().count()
check("REQ-110", "o nº de ondas observado bate com `n_campaign_waves` da config",
      n_ondas == cfg.n_campaign_waves, f"{n_ondas} ondas == cfg.n_campaign_waves={cfg.n_campaign_waves}")

<a id="s4"></a>
## S4 — Atribuição temporal (REQ-103, G4)

**Regra.** Toda transação atribuída cai em `[view_time, valid_until]` — depois do view e
dentro da validade — e cada transação disputada pertence a exatamente uma oferta.
**Prova.** Asserts na base completa, dois casos-limite reais, reconciliação da disputa com o
log de premissa do pipeline, e prova de determinismo por dupla execução.

In [10]:
SEC = "S4"
convertidas = labeled.filter(F.col("assigned_txn_count") > 0)

check("G4", "toda linha com transação atribuída tem view (a atribuição exige view precedente)",
      convertidas.filter(F.col("view_time").isNull()).count() == 0, "0 atribuições sem view")
check("G4", "a primeira transação atribuída nunca precede o view",
      convertidas.filter(F.col("first_assigned_txn_time") < F.col("view_time")).count() == 0,
      "0 transações atribuídas antes do view")
check("G4", "a primeira transação atribuída nunca ultrapassa a validade",
      convertidas.filter(F.col("first_assigned_txn_time") > F.col("valid_until")).count() == 0,
      "0 transações atribuídas após valid_until")
check("G4", "a primeira transação atribuída nunca precede o recebimento",
      convertidas.filter(F.col("first_assigned_txn_time") < F.col("received_time")).count() == 0,
      "0 transações atribuídas antes do received_time")

# Prova forte: TODA transação candidata está na janela pós-view, por construção do join.
fora = candidates.filter(
    (F.col("txn_time") < F.col("view_time")) | (F.col("txn_time") > F.col("valid_until"))
).count()
check("G4", "nenhuma transação candidata está fora de [view_time, valid_until]",
      fora == 0, f"{candidates.count():,} candidatas, 0 fora da janela")

In [11]:
# Caso-limite real: transação fora da janela não é atribuída.
exemplo_fora = attributed.join(txns, on="account_id", how="inner").filter(
    (F.col("txn_time") > F.col("valid_until")) & (F.col("assigned_txn_count") == 0)
).select("account_id", "offer_id", "received_time", "valid_until", "txn_time", "assigned_txn_count").limit(1)
linha = exemplo_fora.first()

check("REQ-103", "transação posterior à validade não é atribuída (caso real)",
      linha is not None and linha["txn_time"] > linha["valid_until"] and linha["assigned_txn_count"] == 0,
      f"cliente {linha['account_id'][:8]}…: txn em t={linha['txn_time']} > valid_until={linha['valid_until']}")
exemplo_fora.toPandas()

,account_id,offer_id,received_time,valid_until,txn_time,assigned_txn_count
0,25c906289d154b66bf579693f89481c9,2906b810c7d4411798c6938adc9daaa5,0.0,7.0,23.0,0


In [12]:
# Caso-limite real: cliente SEM nenhum evento `transaction` sobrevive ao left join.
sem_txn = "8ec6ce2a7e7949b1bf142def7d0e0586"
n_txn_cliente = events.filter((F.col("account_id") == sem_txn) & (F.col("event") == "transaction")).count()
linhas_cliente = processed.filter(F.col("account_id") == sem_txn)
n_linhas = linhas_cliente.count()
n_zeradas = linhas_cliente.filter(F.col("converted") == 0).count()

check("REQ-103", "cliente sem nenhuma transação mantém suas linhas no grão, zeradas",
      n_txn_cliente == 0 and n_linhas > 0 and n_zeradas == n_linhas,
      f"cliente {sem_txn[:8]}…: 0 transações, {n_linhas} linhas preservadas, todas converted=0")
labeled.filter(F.col("account_id") == sem_txn).select(
    "offer_id", "received_time", "view_time", "assigned_txn_count", "converted").toPandas()

,offer_id,received_time,view_time,assigned_txn_count,converted
0,4d5c57ea9a6940dd891ad53e9dbe8da0,7.0,9.00,0,0
1,2906b810c7d4411798c6938adc9daaa5,17.0,19.25,0,0
2,3f207df678b143eea3cee63160fa8bed,14.0,15.25,0,0
3,fafdcd668e3743c1bb461111dcafc2a4,21.0,21.75,0,0
4,fafdcd668e3743c1bb461111dcafc2a4,0.0,0.50,0,0


In [13]:
# Disputa (violação da Premissa 1): transação na janela de mais de uma oferta.
disputadas = (
    candidates.withColumn("n_ofertas", F.count("offer_id").over(Window.partitionBy("account_id", "row_id")))
    .filter(F.col("n_ofertas") > 1)
)
n_disputadas = disputadas.select("account_id", "row_id").distinct().count()

# O pipeline logou o mesmo número? (cross-check do log de premissa)
logados = {int(m) for msg in _capture.messages for m in re.findall(r"em (\d+) transação", msg)}
check("REQ-103", "o nº de transações disputadas bate com o log de premissa do pipeline",
      n_disputadas in logados, f"auditoria={n_disputadas:,}, log do pipeline={sorted(logados)}")

# Exclusividade na disputa. A identidade de contagem `Σ assigned == |elegíveis|`
# É a prova: uma transação disputada atribuída a duas ofertas somaria 2 em Σ assigned
# contra 1 em |elegíveis|; uma perdida somaria 0. A igualdade só vale se cada
# elegível — disputada ou não — tiver exatamente um dono.
check("REQ-103", "cada transação disputada tem exatamente um dono (nem em dobro, nem perdida)",
      n_eligible == n_assigned,
      f"{n_disputadas:,} disputadas; Σ assigned={n_assigned:,} == elegíveis={n_eligible:,}")
print(f"transações disputadas (Premissa 1 violada): {n_disputadas:,}")
print(f"prioridade aplicada: {cfg.attribution_priority.value}")

transações disputadas (Premissa 1 violada): 13,928
prioridade aplicada: earliest_received


In [14]:
# Determinismo: duas execuções independentes produzem atribuição idêntica,
# inclusive nos empates de received_time dentro da mesma onda.
colunas = ["account_id", "offer_id", "received_time", "view_time", "assigned_txn_count", "assigned_txn_amount_sum"]
r1 = attribute(events, offers, cfg).select(*colunas)
r2 = attribute(events, offers, cfg).select(*colunas)
diferenca = r1.exceptAll(r2).count() + r2.exceptAll(r1).count()

check("REQ-103", "a atribuição é determinística entre execuções (desempate estável)",
      diferenca == 0, f"diferença simétrica = {diferenca} linhas")

# Honestidade sobre a cobertura: o empate de `received_time` (duas ofertas ao mesmo
# cliente no mesmo instante) simplesmente NÃO ocorre neste dataset. O desempate por
# `offer_id` é defensivo; quem o exercita é a fixture sintética em
# tests/test_attribution.py::test_identical_received_time_resolves_deterministically_by_offer_id
empates = (
    processed.groupBy("account_id", "received_time").agg(F.count("*").alias("ofertas_no_mesmo_instante"))
    .filter(F.col("ofertas_no_mesmo_instante") > 1).count()
)
print(f"clientes×instantes com mais de uma oferta no mesmo received_time: {empates:,}")
print("→ o caso de empate não existe no dado real; o desempate estável é coberto por teste sintético.")

clientes×instantes com mais de uma oferta no mesmo received_time: 0
→ o caso de empate não existe no dado real; o desempate estável é coberto por teste sintético.


<a id="s5"></a>
## S5 — Label influence-aware (REQ-104, G3, G5)

**Regra.** `converted=1` ⇒ existe view em `[received_time, valid_until]` **e** transação
atribuída com `txn_time ≥ view_time`; a conversão nunca deriva de `offer completed`.
**Prova.** Assert da ordem completa na base inteira, mais quatro casos-limite reais: compra
pré-view, view simultânea ao recebimento, `completed` sem view, e informational sem `completed`.

In [15]:
SEC = "S5"
conv = labeled.filter(F.col("converted") == 1)
n_conv = conv.count()

check("G3", "converted=1 exige um view atribuído",
      conv.filter(F.col("view_time").isNull()).count() == 0, f"{n_conv:,} conversões, 0 sem view")
check("G3", "o view da conversão está dentro de [received_time, valid_until]",
      conv.filter((F.col("view_time") < F.col("received_time")) | (F.col("view_time") > F.col("valid_until"))).count() == 0,
      "0 views fora da janela de validade")
check("G4", "a transação da conversão ocorre em [view_time, valid_until]",
      conv.filter((F.col("first_assigned_txn_time") < F.col("view_time"))
                  | (F.col("first_assigned_txn_time") > F.col("valid_until"))).count() == 0,
      "0 conversões com transação fora da janela pós-view")
check("REQ-104", "converted=1 ⇔ existe transação atribuída",
      labeled.filter((F.col("converted") == 1) != (F.col("assigned_txn_count") > 0)).count() == 0,
      "label e atribuição concordam em toda linha")
check("REQ-104", "conversion_value é zero exatamente quando não há conversão",
      labeled.filter((F.col("converted") == 0) & (F.col("conversion_value") != 0.0)).count() == 0,
      "0 linhas não-convertidas com valor")
print(f"conversões: {n_conv:,} de {labeled.count():,} recebimentos")

conversões: 37,899 de 76,277 recebimentos


In [16]:
# Caso-limite real: comprou ANTES de ver. É a classe de bug que inflaria o efeito
# (o cliente compraria de todo modo — sure thing), e deve dar converted=0.
janela = labeled.join(txns, on="account_id", how="inner").filter(
    (F.col("txn_time") >= F.col("received_time")) & (F.col("txn_time") <= F.col("valid_until")))
por_recebimento = janela.groupBy(
    "account_id", "offer_id", "received_time", "view_time", "converted"
).agg(
    F.sum((F.col("txn_time") >= F.col("view_time")).cast("int")).alias("txns_pos_view"),
    F.count("*").alias("txns_na_janela"),
)
so_pre_view = por_recebimento.filter(
    F.col("view_time").isNotNull() & (F.col("txns_pos_view") == 0) & (F.col("txns_na_janela") > 0)
).cache()
n_pre = so_pre_view.count()

check("G4", "recebimento cuja única compra na janela é anterior ao view não converte",
      n_pre > 0 and so_pre_view.filter(F.col("converted") == 1).count() == 0,
      f"{n_pre:,} recebimentos com compra só pré-view, todos converted=0")
so_pre_view.select("account_id", "offer_id", "received_time", "view_time",
                   "txns_na_janela", "txns_pos_view", "converted").limit(3).toPandas()

,account_id,offer_id,received_time,view_time,txns_na_janela,txns_pos_view,converted
0,10af934bbfb641a49d9376d61dcb203a,4d5c57ea9a6940dd891ad53e9dbe8da0,0.0,4.0,1,0,0
1,bd9640ac357e463bac65c117ca88d0a3,f19421c1d4aa40978ebb69ca19b0e20d,7.0,8.0,1,0,0
2,ddc6402c0b3f48f285a93ad98c5496db,2906b810c7d4411798c6938adc9daaa5,7.0,14.0,2,0,0


In [17]:
# Caso-limite real: view simultânea ao recebimento (view_time == received_time).
# A fronteira é INCLUSIVA — `view_time >= received_time` — e deve ser aceita.
views_em_t0 = events.filter((F.col("event") == "offer viewed") & (F.col("time") == 0.0)).count()
simultaneas = labeled.filter(F.col("view_time") == F.col("received_time")).cache()
n_sim = simultaneas.count()
n_sim_conv = simultaneas.filter(F.col("converted") == 1).count()

check("REQ-104", "view simultânea ao recebimento é aceita (fronteira inclusiva), não descartada",
      n_sim > 0 and n_sim_conv > 0,
      f"{n_sim:,} recebimentos com view_time==received_time; {n_sim_conv:,} convertem")
print(f"eventos `offer viewed` em t=0.0: {views_em_t0:,}")
print(f"recebimentos com view no mesmo instante: {n_sim:,} (convertem: {n_sim_conv:,})")

eventos `offer viewed` em t=0.0: 2,072
recebimentos com view no mesmo instante: 11,850 (convertem: 8,569)


In [18]:
# Caso-limite real: `offer completed` na janela SEM nenhum view → não converte.
completados = events.filter(F.col("event") == "offer completed").select(
    "account_id", F.col("offer_ref").alias("offer_id"), F.col("time").alias("completed_time"))
completado_sem_view = (
    labeled.filter(F.col("view_time").isNull())
    .join(completados, on=["account_id", "offer_id"], how="inner")
    .filter((F.col("completed_time") >= F.col("received_time"))
            & (F.col("completed_time") <= F.col("valid_until")))
).cache()
n_csv = completado_sem_view.count()

check("G3", "`offer completed` sem view precedente nunca vira conversão (sure thing)",
      n_csv > 0 and completado_sem_view.filter(F.col("converted") == 1).count() == 0,
      f"{n_csv:,} recebimentos completados sem view, todos converted=0")

# Taxa de "completou sem ver" medida no dado (contexto da Premissa 2).
vistos = events.filter(F.col("event") == "offer viewed").select(
    "account_id", F.col("offer_ref").alias("offer_id"), F.col("time").alias("view_time"))
pares = (completados.join(vistos, on=["account_id", "offer_id"], how="left")
         .groupBy("account_id", "offer_id", "completed_time")
         .agg(F.max((F.col("view_time") <= F.col("completed_time")).cast("int")).alias("teve_view")))
tot, sem = pares.count(), pares.filter(F.coalesce(F.col("teve_view"), F.lit(0)) == 0).count()
print(f"`completed` sem view precedente: {sem:,}/{tot:,} = {100*sem/tot:.1f}%")
print("(A Premissa 2 cita 28,4%; nenhuma definição razoável reproduz esse número — ver S9.)")

`completed` sem view precedente: 8,568/33,182 = 25.8%
(A Premissa 2 cita 28,4%; nenhuma definição razoável reproduz esse número — ver S9.)


In [19]:
# G5: informational não tem evento `offer completed` — e ainda assim converte,
# provando que a conversão vem da janela pós-view, nunca do evento.
ids_info = [r["id"] for r in offers.filter(F.col("offer_type") == "informational").select("id").collect()]
n_info_completed = events.filter(
    (F.col("event") == "offer completed") & (F.col("offer_ref").isin(ids_info))).count()
n_info_conv = processed.filter((F.col("offer_type") == "informational") & (F.col("converted") == 1)).count()

check("G5", "não existe nenhum evento `offer completed` para ofertas informational",
      n_info_completed == 0, f"{len(ids_info)} ofertas informational, 0 eventos completed")
check("G5", "informational converte mesmo sem `offer completed` (label vem da janela pós-view)",
      n_info_conv > 0, f"{n_info_conv:,} conversões de informational")

processed.groupBy("offer_type").agg(
    F.count("*").alias("recebimentos"),
    F.sum("converted").alias("conversões"),
    F.round(F.avg("converted"), 4).alias("taxa"),
).toPandas()

,offer_type,recebimentos,conversões,taxa
0,bogo,30499,17172,0.5630
1,discount,30543,16092,0.5269
2,informational,15235,4635,0.3042


<a id="s6"></a>
## S6 — Anti-leakage de features (REQ-105, G2)

**Regra.** Nenhuma feature `hist_*` de uma linha incorpora evento com `time ≥ received_time`
daquela linha.
**Prova.** Um oráculo independente recomputa as features sobre a **base completa** usando só
eventos passados, e assere igualdade exata com a saída do pipeline; depois, um cliente
sintético cujo evento pós-recebimento mudaria a feature se vazasse — assert de que não muda.

In [20]:
SEC = "S6"
# Oráculo independente (expressão trivialmente correta), sobre TODAS as 76.277 linhas.
grao = labeled.select("account_id", "offer_id", "received_time")
txn_cru = events.filter(F.col("event") == "transaction").select(
    "account_id", F.col("time").alias("t"), F.col("amount").alias("a"))

oraculo = (
    grao.join(txn_cru, on="account_id", how="left")
    .filter(F.col("t") < F.col("received_time"))       # SÓ passado, antes de agregar
    .groupBy("account_id", "offer_id", "received_time")
    .agg(F.count("*").cast("int").alias("o_count"),
         F.sum("a").alias("o_total"),
         F.avg("a").alias("o_ticket"))
)
comparado = (
    featured.select("account_id", "offer_id", "received_time",
                    "hist_txn_count", "hist_spend_total", "hist_avg_ticket")
    .join(oraculo, on=["account_id", "offer_id", "received_time"], how="left")
    .withColumn("o_count", F.coalesce("o_count", F.lit(0)))
    .withColumn("o_total", F.coalesce("o_total", F.lit(0.0)))
    .withColumn("o_ticket", F.coalesce("o_ticket", F.lit(0.0)))
).cache()

TOL = 1e-6
d_count = comparado.filter(F.col("hist_txn_count") != F.col("o_count")).count()
d_total = comparado.filter(F.abs(F.col("hist_spend_total") - F.col("o_total")) > TOL).count()
d_ticket = comparado.filter(F.abs(F.col("hist_avg_ticket") - F.col("o_ticket")) > TOL).count()

check("G2", "hist_txn_count bate com a recontagem independente de eventos passados",
      d_count == 0, f"{comparado.count():,} linhas, 0 divergências")
check("G2", "hist_spend_total bate com a resoma independente de eventos passados",
      d_total == 0, "0 divergências")
check("G2", "hist_avg_ticket bate com a remédia independente de eventos passados",
      d_ticket == 0, "0 divergências")

# Nenhum evento no futuro da linha pode ter entrado: se algum tivesse, o oráculo
# (que filtra t < received_time) discordaria. A igualdade acima é a prova.
check("G2", "nenhuma feature hist_* incorpora evento com time ≥ received_time",
      d_count == 0 and d_total == 0 and d_ticket == 0, "oráculo pré-recebimento == pipeline")

In [21]:
# Cliente sintético: uma compra gritante DEPOIS do recebimento. Se vazasse,
# hist_spend_total saltaria de 10 para 1009. Roda o pipeline real sobre o dado sintético.
import json, tempfile

eventos_sinteticos = [
    {"event": "transaction", "account_id": "acc", "time_since_test_start": 1.0,
     "value": {"amount": 10.0, "offer id": None, "offer_id": None, "reward": None}},
    {"event": "offer received", "account_id": "acc", "time_since_test_start": 2.0,
     "value": {"amount": None, "offer id": "off1", "offer_id": None, "reward": None}},
    {"event": "transaction", "account_id": "acc", "time_since_test_start": 5.0,
     "value": {"amount": 999.0, "offer id": None, "offer_id": None, "reward": None}},
]
ofertas_sinteticas = [{"channels": ["email"], "min_value": 10, "duration": 7.0,
                       "id": "off1", "offer_type": "bogo", "discount_value": 10}]

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    (tmp / "transactions.json").write_text(json.dumps(eventos_sinteticos))
    (tmp / "offers.json").write_text(json.dumps(ofertas_sinteticas))
    cfg_s = load(raw_dir=tmp)
    ev_s = parse_events(spark, cfg_s)
    of_s = read_offers(spark, cfg_s)
    lin = build_features(ev_s, build_label(attribute(ev_s, of_s, cfg_s), cfg_s), of_s, cfg_s).first()

check("G2", "compra posterior ao recebimento não entra em hist_spend_total (cliente sintético)",
      lin["hist_spend_total"] == 10.0 and lin["hist_txn_count"] == 1,
      f"hist_spend_total={lin['hist_spend_total']} (vazaria como 1009.0), hist_txn_count={lin['hist_txn_count']}")

<a id="s7"></a>
## S7 — Custo (REQ-106, G6)

**Regra.** `reward_cost > 0` ⇒ `converted=1` ∧ `offer_type ≠ informational`.
**Prova.** Assert de zero violações na base completa, mais a coerência do valor com o catálogo.

In [22]:
SEC = "S7"
violacoes = processed.filter(
    (F.col("reward_cost") > 0)
    & ((F.col("converted") != 1) | (F.col("offer_type") == "informational"))
).count()
check("G6", "reward_cost > 0 ⇒ converted=1 ∧ offer_type ≠ informational",
      violacoes == 0, f"{processed.count():,} linhas, 0 violações")

check("G6", "informational nunca tem custo de recompensa",
      processed.filter((F.col("offer_type") == "informational") & (F.col("reward_cost") != 0.0)).count() == 0,
      "0 informational com custo")
check("G6", "não-convertido nunca tem custo",
      processed.filter((F.col("converted") == 0) & (F.col("reward_cost") != 0.0)).count() == 0,
      "0 não-convertidos com custo")

# O custo é o discount_value do catálogo, não um número inventado.
catalogo = offers.select(F.col("id").alias("offer_id"), F.col("discount_value").alias("cat_desconto"))
incoerentes = (
    processed.filter((F.col("converted") == 1) & (F.col("offer_type") != "informational"))
    .join(catalogo, on="offer_id")
    .filter(F.abs(F.col("reward_cost") - F.col("cat_desconto")) > 1e-9).count()
)
check("REQ-106", "em conversões bogo/discount o custo é exatamente o discount_value do catálogo",
      incoerentes == 0, "0 divergências contra o catálogo")

processed.groupBy("offer_type").agg(
    F.sum("converted").alias("conversões"),
    F.round(F.sum("reward_cost"), 2).alias("custo_total"),
    F.round(F.sum("conversion_value"), 2).alias("receita_atribuída"),
).toPandas()

,offer_type,conversões,custo_total,receita_atribuída
0,bogo,17172,134130.0,534946.84
1,discount,16092,43488.0,681695.99
2,informational,4635,0.0,100543.63


<a id="s8"></a>
## S8 — Contrato e schema (REQ-107, G8)

**Regra.** O dataset processado obedece ao `StructType` do contrato, passa na validação
Pydantic de amostra, e só admite nulo nas colunas declaradas nullable.
**Prova.** `contract.assert_schema` / `assert_no_unexpected_nulls` / `validate_sample` sobre o
dataset completo, mais um assert de nulos por coluna.

> **Nota.** O contrato declara **quatro** colunas nullable, não duas: `age` e
> `credit_card_limit` (perfil ausente, Premissa 3) e `hist_recency_days` /
> `hist_time_view_to_conv`, onde `null` significa *"não há histórico"* — sinal que as árvores
> consomem direto e que fabricar um número (`recency=0` diria "comprou hoje") corromperia em
> silêncio. G8 em `schema-processed.md` foi corrigido para refletir isso.

In [23]:
SEC = "S8"
# As duas formas do contrato saem da mesma lista canônica — não podem divergir.
check("REQ-107", "`StructType` e modelo Pydantic declaram exatamente as mesmas colunas",
      [f.name for f in contract.PROCESSED_SCHEMA.fields] == list(contract.ProcessedRow.model_fields)
      == contract.CONTRACT_COLUMNS, f"{len(contract.CONTRACT_COLUMNS)} colunas no contrato")

contract.assert_schema(processed)              # levanta se nomes/ordem/tipos divergirem
check("REQ-107", "o schema do dataset é idêntico ao StructType do contrato",
      processed.columns == contract.CONTRACT_COLUMNS, "nomes, ordem e tipos conferem")

contract.assert_no_unexpected_nulls(processed)  # levanta se houver nulo indevido
n_amostra = contract.validate_sample(processed, cfg)
check("REQ-107", "a validação Pydantic da amostra não levanta erro",
      n_amostra > 0, f"{n_amostra} linhas validadas (cfg.contract_sample_size)")

In [24]:
# Assert de nulos POR COLUNA, sobre a base completa.
nulos = processed.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in contract.CONTRACT_COLUMNS]
).first().asDict()

nao_nullable = [c for c in contract.CONTRACT_COLUMNS if c not in contract.NULLABLE_COLUMNS]
infratoras = {c: nulos[c] for c in nao_nullable if nulos[c] > 0}
check("G8", "nenhuma coluna não-nullable do contrato contém nulo",
      not infratoras, f"{len(nao_nullable)} colunas não-nullable, 0 com nulo")

pd.DataFrame(
    [{"coluna": c, "nulos": nulos[c], "nullable_no_contrato": c in contract.NULLABLE_COLUMNS}
     for c in contract.CONTRACT_COLUMNS if nulos[c] > 0 or c in contract.NULLABLE_COLUMNS]
)

,coluna,nulos,nullable_no_contrato
0,age,9776,True
1,credit_card_limit,9776,True
2,hist_recency_days,20952,True
3,hist_time_view_to_conv,47850,True


<a id="s9"></a>
## S9 — Veredito

**Regra.** Nenhuma garantia auditada pode falhar.
**Prova.** Tabela de todas as verificações e `assert all(passed)` — qualquer FAIL torna o
notebook vermelho. (Uma violação já teria levantado na célula que a provou; esta é a rede.)

In [25]:
SEC = "S9"
from IPython.display import HTML

veredito = pd.DataFrame(AUDIT)
# Âncora clicável para a seção (e a célula) que provou cada regra.
veredito["seção"] = veredito["seção"].map(lambda s: f'<a href="#{s.lower()}">{s}</a>')
HTML(veredito[["regra", "seção", "afirmação", "status", "evidência"]].to_html(escape=False, index=False))

regra,seção,afirmação,status,evidência
FATO,S0,o JSON bruto tem exatamente os eventos esperados,PASS,"306,534 eventos brutos"
FATO,S0,a composição por tipo de evento é a esperada,PASS,"{'transaction': 138953, 'offer received': 76277, 'offer completed': 33579, 'offer viewed': 57725}"
G1,S0,cada `offer received` gera exatamente uma linha do grão,PASS,"76,277 received → 76,277 linhas no grão"
REQ-103,S0,toda transação elegível é atribuída a exatamente uma oferta,PASS,elegíveis=102326 == Σ assigned=102326
REQ-103,S0,nenhuma transação é atribuída mais de uma vez,PASS,Σ assigned=102326 ≤ total=138953
REQ-101,S1,"`offer received` lê a oferta de `offer id` (com espaço), sempre presente",PASS,"76277 eventos, 0 nulos em `offer id`"
REQ-101,S1,`offer received` nunca usa `offer_id` (underscore),PASS,"76277 eventos, 0 usos de `offer_id`"
REQ-101,S1,"`offer viewed` lê a oferta de `offer id` (com espaço), sempre presente",PASS,"57725 eventos, 0 nulos em `offer id`"
REQ-101,S1,`offer viewed` nunca usa `offer_id` (underscore),PASS,"57725 eventos, 0 usos de `offer_id`"
REQ-101,S1,"`offer completed` lê a oferta de `offer_id` (underscore), sempre presente",PASS,"33579 eventos, 0 nulos em `offer_id`"


In [26]:
resumo = veredito.groupby("status").size().to_dict()
print(f"verificações: {len(AUDIT)} | {resumo}")
print("\nObservações levantadas pela auditoria (não são falhas do pipeline):")
print(f"  • `completed` sem view precedente mede {100*sem/tot:.1f}%; a Premissa 2 cita 28,4%.")
print(f"  • {n_unassigned:,} transações ({100*n_unassigned/n_txn:.1f}%) não pertencem a nenhuma janela pós-view.")

assert all(r["status"] == "PASS" for r in AUDIT), "AUDITORIA VERMELHA: há garantia violada."
print("\n✓ AUDITORIA VERDE — todas as garantias provadas sobre o dado real.")

verificações: 54 | {'PASS': 54}

Observações levantadas pela auditoria (não são falhas do pipeline):
  • `completed` sem view precedente mede 25.8%; a Premissa 2 cita 28,4%.
  • 36,627 transações (26.4%) não pertencem a nenhuma janela pós-view.

✓ AUDITORIA VERDE — todas as garantias provadas sobre o dado real.


In [27]:
spark.stop()